In [0]:
-- =====================================================================
-- CPP Monthly Trend Analysis by Population (EXH / Cineplex Theatre)
-- CPP = Total Concession Revenue / Total Ticket Quantity
-- Populations: CC Members | Non-CC Scene Members | Total Business
-- Population definitions per Cineclub Concession Behaviour (Verified)
-- =====================================================================

WITH monthly_tickets AS (
    SELECT
        DATE_TRUNC('month', TO_DATE(CAST(t.VisitDateId AS STRING), 'yyyyMMdd')) AS visit_month,
        SUM(CASE WHEN t.CineClubFLAG = 'Yes' THEN t.Quantity ELSE 0 END)                                            AS cc_tickets,
        SUM(CASE WHEN t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes' THEN t.Quantity ELSE 0 END)     AS scene_tickets,
        SUM(t.Quantity)                                                                                               AS total_tickets,
        COUNT(DISTINCT CASE WHEN t.CineClubFLAG = 'Yes' AND t.CDE_ID > 0 THEN t.CDE_ID END)                         AS cc_ticket_patrons,
        COUNT(DISTINCT CASE WHEN t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes'
                                  AND t.CDE_ID > 0 THEN t.CDE_ID END)                                                AS scene_ticket_patrons,
        COUNT(DISTINCT CASE WHEN t.CDE_ID > 0 THEN t.CDE_ID END)                                                     AS total_ticket_patrons
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
    WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
      AND t.TicketFLAG = 'Yes'
      AND t.VisitDateId >= 20240101
      AND t.VisitDateId <  CAST(DATE_FORMAT(DATE_TRUNC('month', CURRENT_DATE()), 'yyyyMMdd') AS INT)
    GROUP BY 1
),

monthly_concessions AS (
    SELECT
        DATE_TRUNC('month', TO_DATE(CAST(t.VisitDateId AS STRING), 'yyyyMMdd')) AS visit_month,
        SUM(CASE WHEN t.CineClubFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END)                                          AS cc_concession_rev,
        SUM(CASE WHEN t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes' THEN t.Net_Revenue ELSE 0 END)   AS scene_concession_rev,
        SUM(t.Net_Revenue)                                                                                             AS total_concession_rev,
        COUNT(DISTINCT CASE WHEN t.CineClubFLAG = 'Yes' AND t.CDE_ID > 0 THEN t.CDE_ID END)                          AS cc_concession_patrons,
        COUNT(DISTINCT CASE WHEN t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes'
                                  AND t.CDE_ID > 0 THEN t.CDE_ID END)                                                 AS scene_concession_patrons,
        COUNT(DISTINCT CASE WHEN t.CDE_ID > 0 THEN t.CDE_ID END)                                                      AS total_concession_patrons
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
    WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
      AND t.ConcessionFLAG = 'Yes'
      AND t.VisitDateId >= 20240101
      AND t.VisitDateId <  CAST(DATE_FORMAT(DATE_TRUNC('month', CURRENT_DATE()), 'yyyyMMdd') AS INT)
    GROUP BY 1
)

SELECT
    tk.visit_month,
    YEAR(tk.visit_month)                                                                          AS yr,
    MONTH(tk.visit_month)                                                                         AS mo,

    -- CPP by population
    ROUND(cn.cc_concession_rev    / NULLIF(tk.cc_tickets, 0), 2)                                  AS cc_cpp,
    ROUND(cn.scene_concession_rev / NULLIF(tk.scene_tickets, 0), 2)                               AS scene_cpp,
    ROUND(cn.total_concession_rev / NULLIF(tk.total_tickets, 0), 2)                               AS total_biz_cpp,

    -- CC premium over Scene
    ROUND(
        (cn.cc_concession_rev / NULLIF(tk.cc_tickets, 0))
      - (cn.scene_concession_rev / NULLIF(tk.scene_tickets, 0)), 2
    )                                                                                              AS cc_scene_cpp_gap,

    -- Concession incidence: % of ticket-buying patrons who also bought concessions
    ROUND(cn.cc_concession_patrons    * 100.0 / NULLIF(tk.cc_ticket_patrons, 0), 1)               AS cc_incidence_pct,
    ROUND(cn.scene_concession_patrons * 100.0 / NULLIF(tk.scene_ticket_patrons, 0), 1)            AS scene_incidence_pct,
    ROUND(cn.total_concession_patrons * 100.0 / NULLIF(tk.total_ticket_patrons, 0), 1)            AS total_incidence_pct,

    -- Underlying volumes
    tk.cc_tickets,
    tk.scene_tickets,
    tk.total_tickets,
    ROUND(cn.cc_concession_rev, 2)    AS cc_concession_rev,
    ROUND(cn.scene_concession_rev, 2) AS scene_concession_rev,
    ROUND(cn.total_concession_rev, 2) AS total_concession_rev

FROM monthly_tickets tk
JOIN monthly_concessions cn ON tk.visit_month = cn.visit_month
ORDER BY tk.visit_month